# Lab 0 Task 0.1 — CIFAR-10 CNN Experiments
This notebook implements a simple CNN for CIFAR-10 classification and compares different optimizers and activation functions as required for Task 0.1 of Lab 0.

## Setup & Imports

In [1]:
!nvidia-smi # Take a look at the GPU

Sat Mar 28 11:06:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:1A:00.0 Off |                  N/A |
| 29%   30C    P8              2W /  250W |    3838MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import warnings
import wandb
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from typing import List, Dict, Tuple, Any
from utils import make_loaders, train, validate, fit, evaluate

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Make results reproducible
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

2.11.0+cu130
True
NVIDIA GeForce RTX 2080 Ti
Using device: cuda


## Notebook parameters

In [3]:
LOG_WANDB = False # Notebook level parameter for enabling/disabling wandb logging
NUM_WORKERS = 8
PIN_MEMORY = True

## Data Loading

In [4]:
BATCH_SIZE = 256
VAL_FRACTION = 0.2  # 10% of training set used for validation

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_set = CIFAR10(root="../data", train=True,  download=True, transform=transform)
test_set  = CIFAR10(root="../data", train=False, download=True, transform=transform)

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

train_loader, val_loader, test_loader = make_loaders(
    train_set, test_set,
    loader_kwargs=loader_kwargs,
    val_fraction=VAL_FRACTION,
)

classes: list[str] = train_set.classes
print("Classes:", classes)

Dataset split — train: 40,000  val: 10,000  test: 10,000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## Model Definition

The `activation` argument lets us swap between `LeakyReLU` and `Tanh` without rewriting the class.

In [5]:
class SimpleCNN(nn.Module):
    """
    A simple CNN for CIFAR-10 classification.

    Architecture:
        Conv Block 1 : Conv2d(3  → 32)  → act → MaxPool2d  (32×32 → 16×16)
        Conv Block 2 : Conv2d(32 → 64)  → act → MaxPool2d  (16×16 →  8×8)
        Conv Block 3 : Conv2d(64 → 128) → act → MaxPool2d  ( 8×8  →  4×4)
        FC 1         : Linear(128*4*4 → 256) → act
        FC 2         : Linear(256 → num_classes)
    """

    def __init__(self, num_classes: int = 10, activation: nn.Module = nn.LeakyReLU()) -> None:
        super().__init__()

        self.conv1 = nn.Conv2d(3,  32,  kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.act  = activation
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(self.act(self.conv1(x)))
        x = self.pool(self.act(self.conv2(x)))
        x = self.pool(self.act(self.conv3(x)))
        x = torch.flatten(x, start_dim=1)
        x = self.act(self.fc1(x))
        return self.fc2(x)

## Experiment A – SGD + LeakyReLU
*(The original baseline from the script)*

In [6]:
# Hyperparameters
NUM_EPOCHS = 10
LEARNING_RATE = 1e-2

model_a = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_a = optim.SGD(model_a.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="SGD + LeakyReLU",
    tags=["Task 0.1"],
    config={"optimizer": "SGD", "lr": LEARNING_RATE, "activation": "LeakyReLU",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_a = fit(
    model=model_a, 
    optimizer=optimizer_a,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=val_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS,
    log=LOG_WANDB
)

evaluate(model=model_a, test_loader=test_loader,
         criterion=nn.CrossEntropyLoss(), label="SGD + LeakyReLU")

Epoch  1/10 | train loss 2.3008, train acc 10.43% | val loss 2.2970, val acc 10.29%
Epoch  2/10 | train loss 2.2935, train acc 11.19% | val loss 2.2878, val acc 12.84%
Epoch  3/10 | train loss 2.2775, train acc 15.43% | val loss 2.2615, val acc 16.75%
Epoch  4/10 | train loss 2.2214, train acc 20.40% | val loss 2.1593, val acc 24.09%
Epoch  5/10 | train loss 2.0964, train acc 26.44% | val loss 2.0347, val acc 28.13%
Epoch  6/10 | train loss 1.9814, train acc 29.93% | val loss 1.9323, val acc 30.20%
Epoch  7/10 | train loss 1.9047, train acc 32.09% | val loss 1.8719, val acc 32.77%
Epoch  8/10 | train loss 1.8481, train acc 34.12% | val loss 1.8135, val acc 34.86%
Epoch  9/10 | train loss 1.7975, train acc 35.99% | val loss 1.7543, val acc 37.14%
Epoch 10/10 | train loss 1.7578, train acc 37.45% | val loss 1.8159, val acc 34.54%

Restored best weights (val acc 37.14%)
[SGD + LeakyReLU] Test loss: 1.7575 | Test acc: 37.47%


(1.7575118453979492, 37.47)

## Experiment B – Adam + LeakyReLU
*(Task: swap SGD for Adam and report accuracy)*

In [7]:
# Hyperparameters
NUM_EPOCHS = 10
LEARNING_RATE = 1e-3

model_b     = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_b = optim.Adam(model_b.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="Adam + LeakyReLU",
    tags=["Task 0.1"],
    config={"optimizer": "Adam", "lr": LEARNING_RATE, "activation": "LeakyReLU",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_b = fit(
    model=model_b, 
    optimizer=optimizer_b,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=val_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS,
    log=LOG_WANDB
)

evaluate(model=model_b, test_loader=test_loader,
         criterion=nn.CrossEntropyLoss(), label="Adam + LeakyReLU")

Epoch  1/10 | train loss 1.6011, train acc 41.77% | val loss 1.4197, val acc 49.34%
Epoch  2/10 | train loss 1.2045, train acc 56.91% | val loss 1.1470, val acc 59.01%
Epoch  3/10 | train loss 1.0144, train acc 64.06% | val loss 1.0540, val acc 61.22%
Epoch  4/10 | train loss 0.8868, train acc 68.92% | val loss 0.9355, val acc 66.99%
Epoch  5/10 | train loss 0.7931, train acc 72.22% | val loss 0.8913, val acc 68.46%
Epoch  6/10 | train loss 0.6986, train acc 75.73% | val loss 0.8302, val acc 70.50%
Epoch  7/10 | train loss 0.6249, train acc 78.15% | val loss 0.8127, val acc 71.94%
Epoch  8/10 | train loss 0.5460, train acc 80.95% | val loss 0.8027, val acc 72.33%
Epoch  9/10 | train loss 0.4769, train acc 83.42% | val loss 0.8289, val acc 72.38%
Epoch 10/10 | train loss 0.4112, train acc 85.81% | val loss 0.8553, val acc 73.21%

Restored best weights (val acc 73.21%)
[Adam + LeakyReLU] Test loss: 0.8571 | Test acc: 72.93%


(0.8570613464355469, 72.93)

## Experiment C – Adam + Tanh
*(Task: swap LeakyReLU for Tanh and report accuracy)*

In [8]:
# Hyperparameters
NUM_EPOCHS = 10
LEARNING_RATE = 1e-3

model_c     = SimpleCNN(num_classes=len(classes), activation=nn.Tanh()).to(device)
optimizer_c = optim.Adam(model_c.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="Adam + Tanh",
    tags=["Task 0.1"],
    config={"optimizer": "Adam", "lr": LEARNING_RATE, "activation": "Tanh",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_c = fit(
    model=model_c, 
    optimizer=optimizer_c,
    criterion=nn.CrossEntropyLoss(),
    train_loader=train_loader,
    val_loader=val_loader,
    wandb_kwargs=wandb_kwargs,
    num_epochs=NUM_EPOCHS,
    log=LOG_WANDB
)

evaluate(model=model_c, test_loader=test_loader,
         criterion=nn.CrossEntropyLoss(), label="Adam + Tanh")

Epoch  1/10 | train loss 1.4713, train acc 47.35% | val loss 1.2136, val acc 56.75%
Epoch  2/10 | train loss 1.0835, train acc 61.83% | val loss 1.0535, val acc 62.88%
Epoch  3/10 | train loss 0.9094, train acc 68.18% | val loss 0.9611, val acc 66.38%
Epoch  4/10 | train loss 0.7917, train acc 72.43% | val loss 0.8778, val acc 69.30%
Epoch  5/10 | train loss 0.6778, train acc 76.56% | val loss 0.8648, val acc 70.43%
Epoch  6/10 | train loss 0.5826, train acc 80.06% | val loss 0.8290, val acc 71.65%
Epoch  7/10 | train loss 0.4939, train acc 83.21% | val loss 0.8148, val acc 72.15%
Epoch  8/10 | train loss 0.3967, train acc 87.06% | val loss 0.8279, val acc 72.86%
Epoch  9/10 | train loss 0.3172, train acc 90.30% | val loss 0.8113, val acc 73.93%
Epoch 10/10 | train loss 0.2433, train acc 93.12% | val loss 0.8668, val acc 73.09%

Restored best weights (val acc 73.93%)
[Adam + Tanh] Test loss: 0.8117 | Test acc: 73.82%


(0.811715651512146, 73.82)